# Experiment metrics over time (staged pipeline)

Chronological view of trials under `logs/meta_agent/`:

- `**/experiment_memory.jsonl` — per-iteration runs (1a, 1b refine, 1c aug, …)
- `arch_search_1a_results.json`, `arch_search_1c_results.json`, `arch_search_1d/1e_results.json`

**Primary:** `macro_average_precision` (meta-agent ranking on labeled soundscapes).  
**Also plotted:** `median_per_class_auc`, `competition_macro_auc`.

Helper module: `notebooks/experiment_timeline.py` (reporting only, not in meta-agent).  
CSV: `logs/meta_agent/experiment_timeline.csv`

In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd().resolve()
NOTEBOOKS = ROOT if (ROOT / "experiment_timeline.py").exists() else ROOT / "notebooks"
if not (NOTEBOOKS / "experiment_timeline.py").exists():
    NOTEBOOKS = ROOT.parent / "notebooks"
    ROOT = NOTEBOOKS.parent
else:
    ROOT = NOTEBOOKS.parent
sys.path.insert(0, str(NOTEBOOKS))

import matplotlib.pyplot as plt
import pandas as pd

from experiment_timeline import (
    METRIC_COLUMNS,
    collect_experiment_timeline,
    export_timeline_csv,
)

META_LOGS = ROOT / "logs" / "meta_agent"
EXCLUDE_BACKUP = True
SUCCESSES_ONLY = False  # set True to hide failed trials

df = collect_experiment_timeline(
    META_LOGS,
    exclude_backup=EXCLUDE_BACKUP,
    successes_only=SUCCESSES_ONLY,
)
csv_path = export_timeline_csv(META_LOGS / "experiment_timeline.csv")
print(f"Collected {len(df)} experiments → {csv_path}")
print(f"  success: {df['success'].sum()}  |  tracks: {df['track'].value_counts().to_dict()}")
df.head(10)

In [ ]:
PLOT_METRICS = [
    ("macro_average_precision", "Macro average precision (primary)"),
    ("median_per_class_auc", "Median per-class AUC"),
    ("competition_macro_auc", "Competition macro AUC"),
]

TRACK_COLORS = {"perch": "#2ecc71", "cnn": "#3498db", "birdnet": "#9b59b6", "unknown": "#95a5a6"}
STAGE_MARKERS = {"1a": "o", "1b": "s", "1c": "^", "1d": "D", "1e": "*", "trial": "."}

plot_df = df[df["success"]].copy() if SUCCESSES_ONLY else df.copy()
if plot_df.empty:
    raise RuntimeError("No successful experiments to plot.")

x = plot_df["experiment_id"].values

fig, axes = plt.subplots(len(PLOT_METRICS), 1, figsize=(14, 4 * len(PLOT_METRICS)), sharex=True)
if len(PLOT_METRICS) == 1:
    axes = [axes]

for ax, (col, title) in zip(axes, PLOT_METRICS):
    for track, grp in plot_df.groupby("track", sort=False):
        idx = grp.index
        y = grp[col]
        mask = y.notna()
        if not mask.any():
            continue
        ax.plot(
            grp.loc[mask, "experiment_id"],
            y[mask],
            label=track,
            color=TRACK_COLORS.get(track, "#7f8c8d"),
            alpha=0.35,
            linewidth=1,
        )
    # best-so-far envelope (all tracks)
    series = plot_df[col].astype(float)
    best = series.expanding().max()
    ax.plot(x, best, color="black", linewidth=2.2, label="best so far")
    ax.scatter(
        x,
        series,
        c=[TRACK_COLORS.get(t, "#7f8c8d") for t in plot_df["track"]],
        s=28,
        alpha=0.75,
        edgecolors="white",
        linewidths=0.4,
    )
    ax.set_ylabel(col)
    ax.set_title(title)
    ax.grid(True, alpha=0.3)
    ax.legend(loc="lower right", fontsize=9)

axes[-1].set_xlabel("Experiment index (chronological)")
fig.suptitle("Staged pipeline — metrics over time", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Annotate best macro_AP step
col = "macro_average_precision"
sub = plot_df[plot_df[col].notna()]
if sub.empty:
    print("No macro_AP values.")
else:
    best_i = sub[col].idxmax()
    row = plot_df.loc[best_i]
    print(
        f"Best macro_AP = {row[col]:.5f}\n"
        f"  experiment #{row['experiment_id']}  {row['timestamp']}\n"
        f"  track={row['track']}  stage={row['stage']}  label={row['label']}\n"
        f"  median_AUC={row.get('median_per_class_auc')}\n"
        f"  source={row['source']}"
    )

In [ ]:
# Optional: one subplot per track (macro_AP only)
tracks = [t for t in plot_df["track"].unique() if t != "unknown"]
if tracks:
    fig, axes = plt.subplots(len(tracks), 1, figsize=(14, 3.5 * len(tracks)), sharex=True)
    if len(tracks) == 1:
        axes = [axes]
    col = "macro_average_precision"
    for ax, track in zip(axes, tracks):
        g = plot_df[plot_df["track"] == track]
        ax.plot(range(1, len(g) + 1), g[col], marker="o", markersize=4, color=TRACK_COLORS.get(track))
        ax.set_title(f"{track.upper()} — macro AP")
        ax.set_ylabel(col)
        ax.grid(True, alpha=0.3)
    axes[-1].set_xlabel("Trial index within collected timeline (track-filtered order)")
    plt.tight_layout()
    plt.show()